# 02 — Enriquecimiento y Unificación

Este notebook explora los datos RAW en Snowflake, muestra la integración con Taxi Zones,
la unificación de esquemas Yellow/Green y la normalización de catálogos (vendor, payment_type, rate_code).

**Prerequisito:** Haber ejecutado `01_ingest_raw.ipynb` (datos en `NYC_TAXI_P3.RAW`).

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("NYC_Taxi_Enrichment").getOrCreate()

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}

YEAR_START   = int(os.environ["YEAR_START"])
YEAR_END     = int(os.environ["YEAR_END"])
SERVICES     = os.environ["SERVICES"].split(",")

print(f"Servicios: {SERVICES}")
print(f"Rango: {YEAR_START}–{YEAR_END}")

Servicios: ['yellow', 'green']
Rango: 2015–2025


## 2.1 Exploración de datos RAW

Verificamos la estructura de `TRIPS_RAW` y conteos por servicio.

In [2]:
print("=" * 60)
print("CELDA 2: Conteos generales por servicio y año")
print("=" * 60)

counts_sql = """
SELECT service_type, source_year, COUNT(*) AS total_rows
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY service_type, source_year
ORDER BY service_type, source_year
"""
print(">>> Ejecutando query de conteos...")
df_counts = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", counts_sql).load())
df_counts.show(50, truncate=False)
print("✓ Conteos obtenidos")
print("=" * 60)

CELDA 2: Conteos generales por servicio y año
>>> Ejecutando query de conteos...
+------------+-----------+----------+
|SERVICE_TYPE|SOURCE_YEAR|TOTAL_ROWS|
+------------+-----------+----------+
|green       |2015       |19233765  |
|green       |2016       |16385541  |
|green       |2017       |11737059  |
|green       |2018       |8899718   |
|green       |2019       |6300985   |
|green       |2020       |1734176   |
|green       |2021       |1068755   |
|green       |2024       |56551     |
|yellow      |2015       |146039231 |
|yellow      |2016       |131131805 |
|yellow      |2017       |113500327 |
|yellow      |2018       |102871387 |
|yellow      |2019       |54855873  |
|yellow      |2020       |24649092  |
|yellow      |2021       |30904308  |
|yellow      |2022       |39656098  |
|yellow      |2023       |38310226  |
|yellow      |2024       |41169720  |
|yellow      |2025       |48722602  |
+------------+-----------+----------+

✓ Conteos obtenidos


In [3]:
print("=" * 60)
print("CELDA 3: Esquema de TRIPS_RAW (muestra)")
print("=" * 60)

schema_sql = "SELECT * FROM NYC_TAXI_P3.RAW.TRIPS_RAW LIMIT 5"
print(">>> Consultando esquema y muestra...")
df_sample = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", schema_sql).load())
df_sample.printSchema()
df_sample.show(5, truncate=False)
print("✓ Esquema mostrado")
print("=" * 60)

CELDA 3: Esquema de TRIPS_RAW (muestra)
>>> Consultando esquema y muestra...
root
 |-- VENDORID: decimal(38,0) (nullable = true)
 |-- TPEP_PICKUP_DATETIME: timestamp (nullable = true)
 |-- TPEP_DROPOFF_DATETIME: timestamp (nullable = true)
 |-- LPEP_PICKUP_DATETIME: timestamp (nullable = true)
 |-- LPEP_DROPOFF_DATETIME: timestamp (nullable = true)
 |-- PASSENGER_COUNT: double (nullable = true)
 |-- TRIP_DISTANCE: double (nullable = true)
 |-- RATECODEID: double (nullable = true)
 |-- STORE_AND_FWD_FLAG: string (nullable = true)
 |-- PULOCATIONID: decimal(38,0) (nullable = true)
 |-- DOLOCATIONID: decimal(38,0) (nullable = true)
 |-- PAYMENT_TYPE: decimal(38,0) (nullable = true)
 |-- FARE_AMOUNT: double (nullable = true)
 |-- EXTRA: double (nullable = true)
 |-- MTA_TAX: double (nullable = true)
 |-- TIP_AMOUNT: double (nullable = true)
 |-- TOLLS_AMOUNT: double (nullable = true)
 |-- IMPROVEMENT_SURCHARGE: double (nullable = true)
 |-- TOTAL_AMOUNT: double (nullable = true)
 |-- CONGE

## 2.2 Integración con Taxi Zone Lookup

La tabla `TAXI_ZONE_LOOKUP` mapea `LocationID` a nombre de zona y borough.
Se usa para enriquecer pickup y dropoff con información geográfica legible.

In [4]:
print("=" * 60)
print("CELDA 4: Verificar TAXI_ZONE_LOOKUP")
print("=" * 60)

zones_sql = "SELECT * FROM NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP ORDER BY LocationID"
print(">>> Consultando zonas...")
df_zones = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", zones_sql).load())
print(f"✓ Total zonas: {df_zones.count()}")
df_zones.show(10, truncate=False)
print("=" * 60)

CELDA 4: Verificar TAXI_ZONE_LOOKUP
>>> Consultando zonas...
✓ Total zonas: 265
+----------+-------------+-----------------------+------------+
|LOCATIONID|BOROUGH      |ZONE                   |SERVICE_ZONE|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows

In [5]:
print("=" * 60)
print("CELDA 5: Ejemplo JOIN trips + zonas")
print("=" * 60)

join_sql = """
SELECT
    r.service_type,
    r.PULocationID,
    pz.Zone    AS pu_zone,
    pz.Borough AS pu_borough,
    r.DOLocationID,
    dz.Zone    AS do_zone,
    dz.Borough AS do_borough,
    r.trip_distance,
    r.total_amount
FROM NYC_TAXI_P3.RAW.TRIPS_RAW r
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP pz ON r.PULocationID = pz.LocationID
LEFT JOIN NYC_TAXI_P3.RAW.TAXI_ZONE_LOOKUP dz ON r.DOLocationID = dz.LocationID
LIMIT 20
"""
print(">>> Ejecutando JOIN de enriquecimiento...")
df_join = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", join_sql).load())
df_join.show(20, truncate=False)
print("✓ JOIN completado")
print("=" * 60)

CELDA 5: Ejemplo JOIN trips + zonas
>>> Ejecutando JOIN de enriquecimiento...
+------------+------------+-----------------------------+----------+------------+-----------------------------+----------+-------------+------------+
|SERVICE_TYPE|PULOCATIONID|PU_ZONE                      |PU_BOROUGH|DOLOCATIONID|DO_ZONE                      |DO_BOROUGH|TRIP_DISTANCE|TOTAL_AMOUNT|
+------------+------------+-----------------------------+----------+------------+-----------------------------+----------+-------------+------------+
|yellow      |142         |Lincoln Square East          |Manhattan |238         |Upper West Side North        |Manhattan |1.55         |8.8         |
|yellow      |114         |Greenwich Village South      |Manhattan |113         |Greenwich Village North      |Manhattan |0.5          |6.96        |
|yellow      |234         |Union Sq                     |Manhattan |255         |Williamsburg (North Side)    |Brooklyn  |4.5          |23.75       |
|yellow      |237     

## 2.3 Unificación Yellow + Green

Yellow usa `tpep_pickup_datetime` / `tpep_dropoff_datetime`.  
Green usa `lpep_pickup_datetime` / `lpep_dropoff_datetime`.  

Se unifican con `COALESCE` para crear columnas comunes `pickup_datetime` y `dropoff_datetime`.
Ambos servicios ya están en la misma tabla `TRIPS_RAW` distinguidos por `service_type`.

In [6]:
print("=" * 60)
print("CELDA 6: Unificación timestamps Yellow/Green")
print("=" * 60)

unify_sql = """
SELECT
    service_type,
    tpep_pickup_datetime,
    lpep_pickup_datetime,
    COALESCE(tpep_pickup_datetime, lpep_pickup_datetime)   AS pickup_datetime,
    tpep_dropoff_datetime,
    lpep_dropoff_datetime,
    COALESCE(tpep_dropoff_datetime, lpep_dropoff_datetime) AS dropoff_datetime
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
WHERE tpep_pickup_datetime IS NOT NULL OR lpep_pickup_datetime IS NOT NULL
LIMIT 10
"""
print(">>> Ejecutando unificación COALESCE...")
df_unify = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", unify_sql).load())
df_unify.show(10, truncate=False)
print("✓ Unificación demostrada")
print("=" * 60)

CELDA 6: Unificación timestamps Yellow/Green
>>> Ejecutando unificación COALESCE...
+------------+--------------------+--------------------+-------------------+---------------------+---------------------+-------------------+
|SERVICE_TYPE|TPEP_PICKUP_DATETIME|LPEP_PICKUP_DATETIME|PICKUP_DATETIME    |TPEP_DROPOFF_DATETIME|LPEP_DROPOFF_DATETIME|DROPOFF_DATETIME   |
+------------+--------------------+--------------------+-------------------+---------------------+---------------------+-------------------+
|yellow      |2015-04-01 00:12:55 |NULL                |2015-04-01 00:12:55|2015-04-01 00:19:41  |NULL                 |2015-04-01 00:19:41|
|yellow      |2015-04-21 07:01:44 |NULL                |2015-04-21 07:01:44|2015-04-21 07:06:45  |NULL                 |2015-04-21 07:06:45|
|yellow      |2015-04-01 00:46:38 |NULL                |2015-04-01 00:46:38|2015-04-01 01:10:14  |NULL                 |2015-04-01 01:10:14|
|yellow      |2015-04-21 07:47:00 |NULL                |2015-04-21 07:

## 2.4 Normalización de catálogos

Se decodifican los campos numéricos a nombres legibles:
- `VendorID` → `vendor_name`
- `payment_type` → `payment_type_desc`
- `RatecodeID` → `rate_code_desc`

In [7]:
print("=" * 60)
print("CELDA 7: Decodificación de VendorID")
print("=" * 60)

vendor_sql = """
SELECT
    VendorID,
    CASE VendorID
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'VeriFone Inc.'
        ELSE 'Unknown'
    END AS vendor_name,
    COUNT(*) AS trips
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY VendorID
ORDER BY trips DESC
"""
print(">>> Consultando vendors...")
df_vendor = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", vendor_sql).load())
df_vendor.show(truncate=False)
print("✓ Vendors decodificados")
print("=" * 60)

CELDA 7: Decodificación de VendorID
>>> Consultando vendors...
+--------+----------------------------+---------+
|VENDORID|VENDOR_NAME                 |TRIPS    |
+--------+----------------------------+---------+
|2       |VeriFone Inc.               |522917695|
|1       |Creative Mobile Technologies|316327529|
|4       |Unknown                     |748092   |
|7       |Unknown                     |536131   |
|6       |Unknown                     |263489   |
|3       |Unknown                     |10297    |
|5       |Unknown                     |1102     |
+--------+----------------------------+---------+

✓ Vendors decodificados


In [8]:
print("=" * 60)
print("CELDA 8: Decodificación de payment_type")
print("=" * 60)

payment_sql = """
SELECT
    payment_type,
    CASE payment_type
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Other'
    END AS payment_type_desc,
    COUNT(*) AS trips
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY payment_type
ORDER BY trips DESC
"""
print(">>> Consultando payment types...")
df_pay = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", payment_sql).load())
df_pay.show(truncate=False)
print("✓ Payment types decodificados")
print("=" * 60)

CELDA 8: Decodificación de payment_type
>>> Consultando payment types...
+------------+-----------------+---------+
|PAYMENT_TYPE|PAYMENT_TYPE_DESC|TRIPS    |
+------------+-----------------+---------+
|1           |Credit card      |560997730|
|2           |Cash             |249163332|
|0           |Other            |21024182 |
|3           |No charge        |4089710  |
|4           |Dispute          |3808132  |
|NULL        |Other            |1718280  |
|5           |Unknown          |2969     |
+------------+-----------------+---------+

✓ Payment types decodificados


In [9]:
print("=" * 60)
print("CELDA 9: Decodificación de RatecodeID")
print("=" * 60)

rate_sql = """
SELECT
    RatecodeID::INTEGER AS rate_code_id,
    CASE RatecodeID::INTEGER
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau/Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END AS rate_code_desc,
    COUNT(*) AS trips
FROM NYC_TAXI_P3.RAW.TRIPS_RAW
GROUP BY RatecodeID::INTEGER
ORDER BY trips DESC
"""
print(">>> Consultando rate codes...")
df_rate = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", rate_sql).load())
df_rate.show(truncate=False)
print("✓ Rate codes decodificados")
print("=" * 60)

CELDA 9: Decodificación de RatecodeID
>>> Consultando rate codes...
+------------+------------------+---------+
|RATE_CODE_ID|RATE_CODE_DESC    |TRIPS    |
+------------+------------------+---------+
|1           |Standard rate     |789442467|
|NULL        |Unknown           |22742462 |
|2           |JFK               |19338352 |
|5           |Negotiated fare   |5152412  |
|3           |Newark            |1737542  |
|99          |Unknown           |1658644  |
|4           |Nassau/Westchester|725310   |
|6           |Group ride        |7146     |
+------------+------------------+---------+

✓ Rate codes decodificados


## 2.5 Auditoría de ingesta

Revisamos `INGESTION_LOG` para confirmar el estado de cada mes/servicio cargado.

In [10]:
print("=" * 60)
print("CELDA 10: Auditoría — INGESTION_LOG")
print("=" * 60)

log_sql = """
SELECT run_id, service_type, source_year, source_month,
       rows_loaded, status, started_at_utc, finished_at_utc
FROM NYC_TAXI_P3.RAW.INGESTION_LOG
ORDER BY service_type, source_year, source_month
"""
print(">>> Consultando log de auditoría...")
df_log = (spark.read.format("snowflake")
    .options(**SF_OPTIONS).option("query", log_sql).load())
print(f"✓ Registros de auditoría: {df_log.count()}")
df_log.show(50, truncate=False)
print("✓ NB02 completado — Enriquecimiento y Unificación")
print("=" * 60)

CELDA 10: Auditoría — INGESTION_LOG
>>> Consultando log de auditoría...
✓ Registros de auditoría: 279
+----------------+------------+-----------+------------+-----------+------+-------------------+-------------------+
|RUN_ID          |SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|ROWS_LOADED|STATUS|STARTED_AT_UTC     |FINISHED_AT_UTC    |
+----------------+------------+-----------+------------+-----------+------+-------------------+-------------------+
|backfill-2025-04|green       |2015       |1           |1508493    |loaded|2026-04-04 03:09:37|2026-04-04 03:10:25|
|backfill-2025-04|green       |2015       |2           |1574830    |loaded|2026-04-04 03:09:52|2026-04-04 03:10:35|
|backfill-2025-04|green       |2015       |3           |1722574    |loaded|2026-04-04 03:10:09|2026-04-04 03:10:53|
|backfill-2025-04|green       |2015       |4           |1664394    |loaded|2026-04-04 03:10:17|2026-04-04 03:11:02|
|backfill-2025-04|green       |2015       |5           |1786848    |loaded|2026-04-04 

## Resumen

- **Taxi Zone Lookup** integrada: permite JOINs por `PULocationID` / `DOLocationID` → zona, borough.
- **Unificación Yellow/Green**: `COALESCE(tpep_*, lpep_*)` crea timestamps unificados.
- **Catálogos normalizados**: VendorID, payment_type, RatecodeID decodificados a nombres legibles.
- Todo listo para construir la OBT en `03_construccion_obt.ipynb`.